# E-Commerce KNN (Fortgeschritten!)

Hier heben wir unser Modell auf das nächste Level. 
1. Wir nutzen **One-Hot-Encoding** um alle Text-Daten (Monate, Kundentypen) nutzbar zu machen.
2. Wir nutzen **GridSearchCV**, um den Computer selbst die beste Architektur für das Netz finden zu lassen.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix

# 1. Echte Daten laden
df = pd.read_csv("online_shoppers_intention.csv")
print(f"Daten geladen: {len(df)} Zeilen")

### 1. Kategoriale Daten umwandeln (One-Hot-Encoding)
Spalten wie `Month` (z.B. 'Feb', 'Nov') oder `VisitorType` ('New_Visitor') kann die KI nicht als Text lesen. Mit `pd.get_dummies()` erzeugen wir für jeden Text-Wert eine eigene Ja/Nein-Spalte (0 oder 1).

In [ ]:
# Wandle Text-Spalten in 0/1 Spalten um
df_encoded = pd.get_dummies(df, columns=['Month', 'VisitorType', 'Weekend'], drop_first=True)

print("So sehen die Daten NACH dem Encoding aus (Achte auf die neuen Spalten ganz rechts!):")
display(df_encoded.head())

# Jetzt nehmen wir ALLE verfügbaren Spalten (außer unser Target 'Revenue')
X = df_encoded.drop(columns=['Revenue'])
y = df_encoded['Revenue'].astype(int)

# Wir haben jetzt viel mehr Spalten (Features) als vorher!
print(f"\nAnzahl der Features für das Training: {X.shape[1]}")

In [ ]:
# Train-Test-Split und Skalieren (wie immer extrem wichtig!)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 2. Hyperparameter-Tuning (GridSearchCV)
Wir müssen nicht mehr raten, wie groß das Netz sein soll. Wir geben Optionen vor, und Scikit-Learn probiert sie aus. (Das Training dauert jetzt ein paar Sekunden länger, weil mehrere Netze gebaut werden!)

In [ ]:
# Unser Basis-Modell
mlp = MLPClassifier(max_iter=500, random_state=42)

# Die Parameter, die der Computer gegeneinander antreten lassen soll:
parameter_gitter = {
    'hidden_layer_sizes': [(16, 8), (32, 16)], # Probiert zwei verschiedene Netz-Größen aus
    'activation': ['relu', 'tanh']             # Probiert zwei verschiedene mathematische Rechenwege aus
}

# GridSearchCV baut quasi ein Turnier auf, um den Gewinner zu finden
# cv=3 bedeutet: Er testet jede Kombination 3-mal, um sicherzugehen.
grid_search = GridSearchCV(mlp, parameter_gitter, cv=3, n_jobs=-1, verbose=1)

print("Starte das Tuning-Turnier... (Bitte warten)")
grid_search.fit(X_train_scaled, y_train)

print("\n🏆 TURNIER BEENDET!")
print("Die beste Kombination lautet:", grid_search.best_params_)

In [ ]:
# Das BESTE Modell aus dem Turnier nutzen wir nun für den finalen Test
bestes_modell = grid_search.best_estimator_
vorhersagen = bestes_modell.predict(X_test_scaled)

matrix = confusion_matrix(y_test, vorhersagen)
wahr_nein, falsch_ja, falsch_nein, wahr_ja = matrix.ravel()

print("=== ZEUGNIS FÜR DAS OPTIMIERTE MODELL ===\n")
print(f"Wir haben {len(y_test)} echte Besucher getestet.\n")
print("HIER HAT DIE KI RECHT GEHABT:")
print(f"✔️ {wahr_nein} (Kein Kauf richtig erkannt)")
print(f"✔️ {wahr_ja} (Kauf richtig erkannt)\n")
print("FEHLER:")
print(f"❌ {falsch_ja} (Fehlalarm - dachte kauft)")
print(f"❌ {falsch_nein} (Übersehen - dachte kauft nicht)\n")

genauigkeit = ((wahr_nein + wahr_ja) / len(y_test)) * 100
print(f"Gesamt-Genauigkeit: {genauigkeit:.1f} %")